# Limpeza de `eventos_assinatura.csv`: do log às tabelas

O log tem um acontecimento por linha. Para analisar, ele vira três tabelas, todas em `dados/processados/`:

| Tabela | Uma linha por | Para quê |
|---|---|---|
| `assinaturas_mes.parquet` (**painel mensal**) | assinatura × mês em que esteve ativa (jul/2024 a set/2026) | churn mensal do enunciado e treino do modelo: situação no início do mês e `perdida_no_mes` |
| `assinaturas_resumo_eventos.parquet` | assinatura (liga com `assinaturas_limpa.parquet` por `assinatura_id`) | perfil de cada assinatura no log: renovações, desligamentos, carência, cancelamento, reembolso, chargeback |
| `trials.parquet` | teste grátis | conversão do teste, inclusive das contas que testaram e não assinaram |

Antes delas sai o log limpo, `eventos_assinatura_limpo.parquet`: data em horário de Brasília, `assinatura_id` em cada
evento da assinatura e a marca dos eventos do teste grátis.

**Regras aplicadas** (todas no `decisoes.md`):
1. Data em horário de Brasília: com `Z` é UTC, sem `Z` já está em Brasília.
2. `renovacao_automatica_desligada` sem plano, até 8 dias após o `trial_iniciado`, é evento do teste grátis.
3. Evento até 24h depois da `assinatura_expirada` é da assinatura que acabou e entra no mês da perda.
4. Motivo da saída vem de `motivo_fim` (assinaturas), não de `detalhe`.
5. Renovação automática das assinaturas anteriores ao log: ligada em 01/07/2024, salvo se o primeiro evento de
   renovação delas for um religamento (decisão nova).

Nada em `dados/brutos/` é alterado: o hash é conferido no início e no fim.

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
INICIO_LOG = pd.Timestamp("2024-07-01")
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)   # 01/10/2026 00:00, horário de Brasília
TZ_BR = "America/Sao_Paulo"

ARQUIVOS = ["eventos_assinatura.csv", "usuarios.csv"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

registro = []

def registrar(tabela, passo, n, obs=""):
    registro.append({"tabela": tabela, "passo": passo, "linhas": n, "observacao": obs})
    print(f"[{tabela}] {passo}: {n:,} linhas" + (f" ({obs})" if obs else ""))

TOGGLES = ["renovacao_automatica_desligada", "renovacao_automatica_religada"]

## 0. Leitura

O log como texto; as assinaturas já limpas (notebook 05), que trazem `inicio`, `data_fim` e `motivo_fim`.

In [2]:
eventos = pd.read_csv(BRUTOS / "eventos_assinatura.csv", dtype=str, keep_default_na=False)
usuarios = pd.read_csv(BRUTOS / "usuarios.csv", dtype=str, keep_default_na=False)
assin = pd.read_parquet(PROCESSADOS / "assinaturas_limpa.parquet")

assert len(eventos) == 275_565   # cabeçalho da seção 06 do dicionário
assert len(assin) == 32_583 and assin["user_id"].is_unique
registrar("log", "0. Arquivo bruto", len(eventos))
eventos["evento"].value_counts().to_frame()

[log] 0. Arquivo bruto: 275,565 linhas


,count
evento,
renovacao,132606
trial_iniciado,46996
assinatura_iniciada,27383
renovacao_automatica_desligada,26923
trial_expirado,19143
assinatura_expirada,17830
renovacao_automatica_religada,1798
periodo_carencia,1346
cancelamento_web,1172


## 1. Data em horário de Brasília e contas de teste

In [3]:
def para_brasilia(s):
    utc = s.str.endswith("Z")
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    out[utc] = (pd.to_datetime(s[utc], format="%Y-%m-%dT%H:%M:%SZ", utc=True)
                .dt.tz_convert(TZ_BR).dt.tz_localize(None))
    out[~utc] = pd.to_datetime(s[~utc], format="%Y-%m-%d %H:%M:%S")
    return out

eventos["data_br"] = para_brasilia(eventos["data_hora"])
assert eventos["data_br"].notna().all()
assert eventos["data_br"].min() >= INICIO_LOG and eventos["data_br"].max() < FIM_CORTE

muda_dia = (eventos["data_br"].dt.normalize() != pd.to_datetime(eventos["data_hora"].str[:10])).sum()
print(f"eventos que mudam de dia com o fuso: {muda_dia:,}")

email = usuarios.set_index("user_id")["email"].str.strip().str.lower()
teste = eventos["user_id"].map(email).str.endswith("@vitaliza.com.br")
print(f"eventos de contas de teste: {teste.sum()}")
assert teste.sum() == 0

eventos = eventos.sort_values(["user_id", "data_br", "evento"], kind="stable").reset_index(drop=True)
registrar("log", "1. Data em Brasília", len(eventos), f"{muda_dia:,} mudam de dia; 0 de contas de teste")

eventos que mudam de dia com o fuso: 21,951
eventos de contas de teste: 0


[log] 1. Data em Brasília: 275,565 linhas (21,951 mudam de dia; 0 de contas de teste)


## 2. Eventos do teste grátis

`trial_iniciado` e `trial_expirado` são do teste. O desligamento da renovação sem plano também: acontece
até 8 dias depois do início do teste, e nenhuma dessas pessoas chega a assinar.

In [4]:
inicio_trial = eventos.loc[eventos["evento"] == "trial_iniciado"].set_index("user_id")["data_br"]
assert inicio_trial.index.is_unique

deslig_sem_plano = (eventos["evento"] == "renovacao_automatica_desligada") & (eventos["plano"] == "")
dias_desde_trial = (eventos["data_br"] - eventos["user_id"].map(inicio_trial)).dt.total_seconds() / 86400
deslig_no_trial = deslig_sem_plano & dias_desde_trial.between(0, 8)

print(f"desligamentos sem plano: {deslig_sem_plano.sum():,}; até 8 dias após o trial: {deslig_no_trial.sum():,} "
      f"(observado: {dias_desde_trial[deslig_no_trial].min():.1f} a {dias_desde_trial[deslig_no_trial].max():.1f} dias)")
assert (deslig_sem_plano == deslig_no_trial).all() and deslig_no_trial.sum() == 6_734

eventos["evento_do_trial"] = eventos["evento"].isin(["trial_iniciado", "trial_expirado"]) | deslig_no_trial
assert (eventos["evento_do_trial"] == (eventos["plano"] == "")).all()
registrar("log", "2. Eventos do teste grátis marcados", len(eventos),
          f"{eventos['evento_do_trial'].sum():,} do trial; {(~eventos['evento_do_trial']).sum():,} da assinatura")

desligamentos sem plano: 6,734; até 8 dias após o trial: 6,734 (observado: 1.3 a 7.7 dias)
[log] 2. Eventos do teste grátis marcados: 275,565 linhas (72,873 do trial; 202,692 da assinatura)


## 3. Assinatura de cada evento

Cada conta tem uma assinatura só (notebook 05), então todo evento que não é do teste pertence à assinatura da conta.
A célula confere que nenhum evento da assinatura cai antes do início dela, e que os que vêm depois da expiração
chegam em até 24h (os avisos atrasados do diagnóstico).

In [5]:
eventos = eventos.merge(assin[["user_id", "assinatura_id", "inicio", "data_fim", "inicio_antes_do_log"]],
                        on="user_id", how="left", validate="many_to_one")
da_assinatura = ~eventos["evento_do_trial"]
assert eventos.loc[da_assinatura, "assinatura_id"].notna().all()
eventos.loc[~da_assinatura, "assinatura_id"] = None

antes_do_inicio = da_assinatura & (eventos["data_br"].dt.normalize() < eventos["inicio"])
print(f"eventos da assinatura antes do início: {antes_do_inicio.sum()}")
assert antes_do_inicio.sum() == 0

expiracao = eventos.loc[eventos["evento"] == "assinatura_expirada"].set_index("assinatura_id")["data_br"]
assert expiracao.index.is_unique
horas_apos = (eventos["data_br"] - eventos["assinatura_id"].map(expiracao)).dt.total_seconds() / 3600
eventos["apos_expiracao"] = da_assinatura & (horas_apos > 0)
print(f"eventos depois da expiração: {eventos['apos_expiracao'].sum()} "
      f"(até {horas_apos[eventos['apos_expiracao']].max():.1f}h) -> "
      f"{eventos.loc[eventos['apos_expiracao'], 'evento'].value_counts().to_dict()}")
assert eventos["apos_expiracao"].sum() == 398 and horas_apos[eventos["apos_expiracao"]].max() <= 24

# mês em que o evento conta: o da data em Brasília; os avisos atrasados contam no mês da perda
eventos["mes"] = eventos["data_br"].dt.to_period("M")
eventos.loc[eventos["apos_expiracao"], "mes"] = eventos.loc[eventos["apos_expiracao"], "data_fim"].dt.to_period("M")
registrar("log", "3. Assinatura de cada evento", len(eventos),
          f"{eventos['apos_expiracao'].sum()} avisos atrasados contam no mês da perda")

eventos da assinatura antes do início: 0
eventos depois da expiração: 398 (até 16.6h) -> {'renovacao_automatica_desligada': 211, 'chargeback': 136, 'reembolso': 40, 'cancelamento_web': 11}
[log] 3. Assinatura de cada evento: 275,565 linhas (398 avisos atrasados contam no mês da perda)


## 4. Log limpo

In [6]:
log = eventos[["user_id", "assinatura_id", "data_br", "mes", "evento", "plano", "canal_cobranca", "detalhe",
               "evento_do_trial", "apos_expiracao", "data_hora"]].rename(columns={"data_hora": "data_hora_original"})
log["mes"] = log["mes"].dt.to_timestamp()
log["plano"] = log["plano"].replace("", None)
log["detalhe"] = log["detalhe"].replace("", None)

PROCESSADOS.mkdir(parents=True, exist_ok=True)
log.to_parquet(PROCESSADOS / "eventos_assinatura_limpo.parquet", index=False)
registrar("log", "4. eventos_assinatura_limpo.parquet", len(log))

[log] 4. eventos_assinatura_limpo.parquet: 275,565 linhas


## 5. Renovação automática: estado ao longo do tempo

O estado muda com `renovacao_automatica_desligada` (desliga) e `renovacao_automatica_religada` (liga).
Toda assinatura começa ligada. Para as anteriores ao log, o estado em 01/07/2024 não aparece: fica ligada,
salvo se o primeiro evento de renovação for um religamento (prova de que estava desligada).

Uma `renovacao` cobrada também liga: só há cobrança com a renovação automática ligada. Isso corrige poucos casos
em que o log grava um desligamento e a cobrança seguinte acontece mesmo assim, quase sempre logo depois de um
religamento (os dois avisos parecem gravados em ordem trocada).

O teste da regra é o retrato: no corte, o estado reconstruído tem de bater com `renovacao_automatica` de `assinaturas.csv`.

In [7]:
tg = (eventos.loc[da_assinatura & eventos["evento"].isin(TOGGLES), ["assinatura_id", "data_br", "evento", "inicio_antes_do_log"]]
      .sort_values(["assinatura_id", "data_br"]).reset_index(drop=True))
tg["ligada_depois"] = tg["evento"] == "renovacao_automatica_religada"
tg["anterior"] = tg.groupby("assinatura_id")["evento"].shift()

primeiro = tg.drop_duplicates("assinatura_id")

# estado: desligamento e religamento, mais as renovações cobradas (que provam a renovação ligada)
est = (eventos.loc[da_assinatura & eventos["evento"].isin(TOGGLES + ["renovacao"]), ["assinatura_id", "data_br", "evento"]]
       .sort_values(["assinatura_id", "data_br"]).reset_index(drop=True))
est["ligada_depois"] = est["evento"] != TOGGLES[0]
est["proximo"] = est.groupby("assinatura_id")["evento"].shift(-1)
est["anterior"] = est.groupby("assinatura_id")["evento"].shift()
deslig_seguido_de_cobranca = (est["evento"] == TOGGLES[0]) & (est["proximo"] == "renovacao")

anomalias = {
    "primeiro evento de renovação é religamento (assinatura anterior ao log)":
        ((primeiro["evento"] == TOGGLES[1]) & primeiro["inicio_antes_do_log"]).sum(),
    "primeiro evento de renovação é religamento (assinatura iniciada no log)":
        ((primeiro["evento"] == TOGGLES[1]) & ~primeiro["inicio_antes_do_log"]).sum(),
    "desligamento duas vezes seguidas": (tg["evento"].eq(TOGGLES[0]) & tg["anterior"].eq(TOGGLES[0])).sum(),
    "desligamento seguido de renovação cobrada": deslig_seguido_de_cobranca.sum(),
    "  ... logo depois de um religamento": (deslig_seguido_de_cobranca & est["anterior"].eq(TOGGLES[1])).sum(),
}
display(pd.Series(anomalias, name="ocorrências").to_frame())
assert deslig_seguido_de_cobranca.sum() == 12

pre_desligadas = set(primeiro.loc[(primeiro["evento"] == TOGGLES[1]) & primeiro["inicio_antes_do_log"], "assinatura_id"])
assin["ligada_no_inicio"] = ~assin["assinatura_id"].isin(pre_desligadas)

def ligada_em(chaves, instante_col):
    """Estado da renovação no instante (exclusive) para cada linha de `chaves` (assinatura_id + instante)."""
    q = chaves[["assinatura_id", instante_col]].reset_index().sort_values(instante_col)
    t = est[["assinatura_id", "data_br", "ligada_depois"]].sort_values("data_br")
    r = pd.merge_asof(q, t, left_on=instante_col, right_on="data_br", by="assinatura_id",
                      allow_exact_matches=False, direction="backward").set_index("index").sort_index()
    inicial = chaves["assinatura_id"].map(assin.set_index("assinatura_id")["ligada_no_inicio"])
    return r["ligada_depois"].astype("boolean").fillna(inicial).astype(bool)

no_corte = assin.loc[assin["ativa_no_corte"], ["assinatura_id", "renovacao_automatica"]].copy()
no_corte["fim"] = FIM_CORTE
no_corte["ligada_reconstruida"] = ligada_em(no_corte, "fim")
confere = pd.crosstab(no_corte["renovacao_automatica"], no_corte["ligada_reconstruida"])
display(confere)
divergem = (no_corte["ligada_reconstruida"] != (no_corte["renovacao_automatica"] == "sim")).sum()
print(f"ativas no corte com estado diferente do retrato: {divergem}")
assert divergem == 0

,ocorrências
primeiro evento de renovação é religamento (assinatura anterior ao log),3
primeiro evento de renovação é religamento (assinatura iniciada no log),19
desligamento duas vezes seguidas,6
desligamento seguido de renovação cobrada,12
... logo depois de um religamento,9


ligada_reconstruida,False,True
renovacao_automatica,,
nao,2250,0
sim,0,12503


ativas no corte com estado diferente do retrato: 0


## 6. Tabela 1: painel mensal (`assinaturas_mes.parquet`)

Uma linha por assinatura e mês, do mês de início (ou jul/2024, para as anteriores ao log) até o mês do fim
(ou set/2026, para as ativas). Definições do enunciado:

- `ativa_inicio_mes`: começou antes do dia 1 e não tinha terminado antes do dia 1;
- `perdida_no_mes`: ativa no início do mês e terminou no mês (numerador do churn);
- `encerrou_no_mes`: terminou no mês, mesmo tendo começado nele (ex.: chargeback nos primeiros dias).

As colunas `n_<evento>` contam os eventos da assinatura no mês. `renovacao_ligada_inicio_mes` é o estado no dia 1
(vazio para quem começou no mês). `meses_de_casa` são os meses completos no dia 1.

In [8]:
primeiro_mes = assin["inicio"].dt.to_period("M").clip(lower=INICIO_LOG.to_period("M"))
ultimo_mes = assin["data_fim"].fillna(CORTE).dt.to_period("M")
n_meses = (ultimo_mes - primeiro_mes).apply(lambda d: d.n) + 1

painel = assin.loc[assin.index.repeat(n_meses),
                   ["assinatura_id", "user_id", "plano_atual", "canal_cobranca", "inicio", "data_fim",
                    "motivo_fim", "inicio_antes_do_log"]].copy()
painel["mes"] = primeiro_mes.loc[painel.index].values + painel.groupby(level=0).cumcount().values
painel = painel.reset_index(drop=True)
registrar("painel", "6.1 Uma linha por assinatura e mês", len(painel), f"esperado {n_meses.sum():,}")
assert len(painel) == n_meses.sum()

d0 = painel["mes"].dt.to_timestamp()
d1 = (painel["mes"] + 1).dt.to_timestamp()
painel["ativa_inicio_mes"] = (painel["inicio"] < d0) & (painel["data_fim"].isna() | (painel["data_fim"] >= d0))
painel["iniciou_no_mes"] = (painel["inicio"] >= d0) & (painel["inicio"] < d1)
painel["encerrou_no_mes"] = (painel["data_fim"] >= d0) & (painel["data_fim"] < d1)
painel["perdida_no_mes"] = painel["ativa_inicio_mes"] & painel["encerrou_no_mes"]
painel["motivo_fim"] = painel["motivo_fim"].where(painel["encerrou_no_mes"])
assert (painel["ativa_inicio_mes"] | painel["iniciou_no_mes"]).all()

meses = (painel["mes"] - painel["inicio"].dt.to_period("M")).apply(lambda d: d.n) - (painel["inicio"].dt.day > 1)
painel["meses_de_casa"] = meses.where(painel["ativa_inicio_mes"]).astype("Int64")

painel["d0"] = d0
estado = ligada_em(painel, "d0")
painel["renovacao_ligada_inicio_mes"] = estado.astype("boolean").where(painel["ativa_inicio_mes"])

contagem = (eventos.loc[da_assinatura].groupby(["assinatura_id", "mes", "evento"]).size()
            .unstack(fill_value=0).add_prefix("n_"))
contagem.index = contagem.index.set_names(["assinatura_id", "mes"])
antes = len(painel)
painel = painel.merge(contagem.reset_index(), on=["assinatura_id", "mes"], how="left", validate="one_to_one")
assert len(painel) == antes
cols_n = [c for c in painel.columns if c.startswith("n_")]
painel[cols_n] = painel[cols_n].fillna(0).astype("int64")

# todo evento da assinatura caiu numa linha do painel
assert painel[cols_n].sum().sum() == da_assinatura.sum()
assert painel["assinatura_id"].astype(str).str.cat(painel["mes"].astype(str), sep="|").is_unique
registrar("painel", "6.2 Situação no mês e eventos", len(painel),
          f"{painel[cols_n].sum().sum():,} eventos da assinatura distribuídos, todos em alguma linha")

[painel] 6.1 Uma linha por assinatura e mês: 327,771 linhas (esperado 327,771)


[painel] 6.2 Situação no mês e eventos: 327,771 linhas (202,692 eventos da assinatura distribuídos, todos em alguma linha)


### 6.1 Conferência com o enunciado

O enunciado traz o churn mensal (perdas no mês ÷ ativos no início do mês) e alguns totais. O painel tem de reproduzi-los.

In [9]:
churn = painel.groupby("mes").agg(ativos_inicio=("ativa_inicio_mes", "sum"), perdas=("perdida_no_mes", "sum"),
                                  inicios=("iniciou_no_mes", "sum"))
churn["churn_%"] = (100 * churn["perdas"] / churn["ativos_inicio"]).round(2)
display(churn.T)

media_2025 = churn.loc["2025-01":"2025-12", "churn_%"].mean()
media_2026 = churn.loc["2026-01":"2026-09", "churn_%"].mean()
ativas_fim = (painel.loc[painel["mes"] == "2026-09", "data_fim"].isna()).sum()

prova = pd.DataFrame([
    ("ativos no início de jan/2025", churn.loc["2025-01", "ativos_inicio"], 8_540),
    ("ativos no início de abr/2026", churn.loc["2026-04", "ativos_inicio"], 14_237),
    ("perdas em abr/2026", churn.loc["2026-04", "perdas"], 1_154),
    ("churn de abr/2026 (%)", round(churn.loc["2026-04", "churn_%"], 1), 8.1),
    ("churn de ago/2026 (%)", round(churn.loc["2026-08", "churn_%"], 1), 5.9),
    ("churn de set/2026 (%)", round(churn.loc["2026-09", "churn_%"], 1), 5.0),
    ("ativos no início de set/2026", churn.loc["2026-09", "ativos_inicio"], 14_319),
    ("churn médio de 2025 (%)", round(media_2025, 1), 5.6),
    ("churn médio de jan a set/2026 (%)", round(media_2026, 1), 6.6),
    ("ativas no corte", ativas_fim, 14_753),
], columns=["conferência", "painel", "enunciado"]).set_index("conferência")
prova["bate"] = prova["painel"] == prova["enunciado"]
display(prova)
assert prova["bate"].all()

mes,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
ativos_inicio,5200.00,5738.00,6379.00,6977.00,7597.0,8169.00,8540.00,9418.00,10014.00,10577.00,10937.00,11317.00,11640.00,11811.00,12020.00,12413.00,12771.00,13023.00,13140.0,13728.00,14111.00,14237.00,14119.00,14102.00,14059.00,14032.00,14319.00
perdas,105.00,275.00,320.00,347.00,342.0,380.00,534.00,529.00,552.00,522.00,543.00,562.00,679.00,774.00,704.00,742.00,723.00,695.00,959.0,830.00,989.00,1154.00,1026.00,866.00,948.00,834.00,714.00
inicios,644.00,918.00,927.00,973.00,920.0,756.00,1423.00,1134.00,1122.00,886.00,931.00,888.00,857.00,989.00,1103.00,1106.00,978.00,816.00,1561.0,1219.00,1131.00,1040.00,1021.00,832.00,930.00,1127.00,1151.00
churn_%,2.02,4.79,5.02,4.97,4.5,4.65,6.25,5.62,5.51,4.94,4.96,4.97,5.83,6.55,5.86,5.98,5.66,5.34,7.3,6.05,7.01,8.11,7.27,6.14,6.74,5.94,4.99


,painel,enunciado,bate
conferência,,,
ativos no início de jan/2025,8540.0,8540.0,True
ativos no início de abr/2026,14237.0,14237.0,True
perdas em abr/2026,1154.0,1154.0,True
churn de abr/2026 (%),8.1,8.1,True
churn de ago/2026 (%),5.9,5.9,True
churn de set/2026 (%),5.0,5.0,True
ativos no início de set/2026,14319.0,14319.0,True
churn médio de 2025 (%),5.6,5.6,True
churn médio de jan a set/2026 (%),6.6,6.6,True


### 6.2 Julho de 2024 tem perdas a menos

O churn de jul/2024 (2,0%) é menos da metade do dos meses seguintes. As perdas por semana sobem de 2 na primeira
semana do log até o ritmo normal (~55) no fim do mês: as saídas cuja decisão aconteceu antes de 01/07/2024 não
aparecem. Jul/2024 fica fora das séries de churn e do treino do modelo (o enunciado também começa em 2025).

In [10]:
fins = assin.loc[assin["data_fim"].between("2024-07-01", "2024-08-31"), "data_fim"]
fins.dt.to_period("W").value_counts().sort_index().rename("perdas na semana").to_frame().T

data_fim,2024-07-01/2024-07-07,2024-07-08/2024-07-14,2024-07-15/2024-07-21,2024-07-22/2024-07-28,2024-07-29/2024-08-04,2024-08-05/2024-08-11,2024-08-12/2024-08-18,2024-08-19/2024-08-25,2024-08-26/2024-09-01
perdas na semana,2,13,26,43,54,54,71,57,63


## 7. Tabela 2: resumo por assinatura (`assinaturas_resumo_eventos.parquet`)

Uma linha por assinatura, com o que o log mostra dela. Para as 5.200 anteriores ao log, as contagens valem só
de 01/07/2024 em diante (`inicio_antes_do_log`).

In [11]:
ev = eventos.loc[da_assinatura]
porr = ev.groupby(["assinatura_id", "evento"]).size().unstack(fill_value=0).add_prefix("n_")
datas = ev.pivot_table(index="assinatura_id", columns="evento", values="data_br", aggfunc=["min", "max"])

resumo = assin[["assinatura_id", "user_id", "inicio_antes_do_log", "ativa_no_corte"]].set_index("assinatura_id")
resumo = resumo.join(porr).fillna({c: 0 for c in porr.columns})
resumo[porr.columns] = resumo[porr.columns].astype("int64")

resumo["primeiro_desligamento"] = datas[("min", "renovacao_automatica_desligada")]
resumo["ultimo_desligamento"] = datas[("max", "renovacao_automatica_desligada")]
resumo["data_carencia"] = datas[("min", "periodo_carencia")]
resumo["data_cancelamento_web"] = datas[("min", "cancelamento_web")]
resumo["data_expiracao"] = datas[("max", "assinatura_expirada")]

# desligamento que levou à expiração: o último antes dela
deslig = ev.loc[ev["evento"] == TOGGLES[0], ["assinatura_id", "data_br"]]
deslig = deslig.merge(expiracao.rename("exp").reset_index(), on="assinatura_id", how="inner")
deslig = deslig[deslig["data_br"] <= deslig["exp"]]
ult = deslig.groupby("assinatura_id").agg(ult=("data_br", "max"), exp=("exp", "first"))
resumo["dias_desligamento_ate_expiracao"] = ((ult["exp"] - ult["ult"]).dt.total_seconds() / 86400).round(2)

corte = resumo.reset_index()[["assinatura_id"]].assign(fim=FIM_CORTE)
corte.index = resumo.index
resumo["renovacao_ligada_no_corte"] = ligada_em(corte.reset_index(drop=True), "fim").values & resumo["ativa_no_corte"].values

resumo["teve_trial_no_log"] = resumo["user_id"].isin(inicio_trial.index)
resumo["trial_inicio"] = resumo["user_id"].map(inicio_trial)
resumo = resumo.reset_index()

assert len(resumo) == len(assin) and resumo["assinatura_id"].is_unique
assert resumo[porr.columns].sum().sum() == da_assinatura.sum()
assert (resumo["teve_trial_no_log"] == ~resumo["inicio_antes_do_log"]).all()
registrar("resumo", "7. Uma linha por assinatura", len(resumo))
display(resumo.describe(include="all").T[["count", "mean", "min", "max"]])

[resumo] 7. Uma linha por assinatura: 32,583 linhas


,count,mean,min,max
assinatura_id,32583,NaN,NaN,NaN
user_id,32583,NaN,NaN,NaN
inicio_antes_do_log,32583,NaN,NaN,NaN
ativa_no_corte,32583,NaN,NaN,NaN
n_assinatura_expirada,32583.0,0.547218,0.0,1.0
n_assinatura_iniciada,32583.0,0.840408,0.0,1.0
n_cancelamento_web,32583.0,0.03597,0.0,2.0
n_chargeback,32583.0,0.009084,0.0,1.0
n_periodo_carencia,32583.0,0.04131,0.0,1.0
n_reembolso,32583.0,0.00221,0.0,1.0


## 8. Tabela 3: testes grátis (`trials.parquet`)

Cada conta faz no máximo um teste. Desfecho: `converteu` (`assinatura_iniciada`), `expirou` (`trial_expirado`)
ou `em_andamento` (sem desfecho até o corte, só testes dos últimos dias).

In [12]:
tr = eventos.loc[eventos["evento"] == "trial_iniciado", ["user_id", "data_br", "canal_cobranca"]].rename(
    columns={"data_br": "trial_inicio", "canal_cobranca": "canal_trial"}).set_index("user_id")
expirou = eventos.loc[eventos["evento"] == "trial_expirado"].set_index("user_id")["data_br"]
inicio_ass = eventos.loc[eventos["evento"] == "assinatura_iniciada"].set_index("user_id")
deslig_trial = eventos.loc[eventos["evento_do_trial"] & (eventos["evento"] == TOGGLES[0])].set_index("user_id")["data_br"]
assert expirou.index.is_unique and inicio_ass.index.is_unique and deslig_trial.index.is_unique
assert not expirou.index.isin(inicio_ass.index).any()   # ninguém expira e assina

tr["desligou_renovacao_no_trial"] = tr.index.isin(deslig_trial.index)
tr["data_desligamento"] = deslig_trial
tr["desfecho"] = np.select([tr.index.isin(inicio_ass.index), tr.index.isin(expirou.index)],
                           ["converteu", "expirou"], "em_andamento")
tr["data_desfecho"] = inicio_ass["data_br"].combine_first(expirou)
tr["dias_de_trial"] = ((tr["data_desfecho"] - tr["trial_inicio"]).dt.total_seconds() / 86400).round(2)
tr["assinatura_id"] = tr.index.map(assin.set_index("user_id")["assinatura_id"]).where(tr["desfecho"] == "converteu")
tr["plano_assinado"] = inicio_ass["plano"]
tr["canal_assinatura"] = inicio_ass["canal_cobranca"]
tr = tr.reset_index()

em_aberto = tr.loc[tr["desfecho"] == "em_andamento", "trial_inicio"]
print(f"testes em andamento: {len(em_aberto)}, iniciados a partir de {em_aberto.min()}")
assert (em_aberto > FIM_CORTE - pd.Timedelta(days=8)).all()
assert not (tr["desligou_renovacao_no_trial"] & (tr["desfecho"] == "converteu")).any()
mudou_canal = (tr["desfecho"].eq("converteu") & (tr["canal_trial"] != tr["canal_assinatura"])).sum()
print(f"assinaturas com canal diferente do teste: {mudou_canal}")

display(pd.crosstab(tr["desfecho"], tr["desligou_renovacao_no_trial"], margins=True, margins_name="total"))
assert len(tr) == 46_996 and (tr["desfecho"] == "converteu").sum() == 27_383
registrar("trials", "8. Uma linha por teste grátis", len(tr),
          f"{(tr['desfecho'] == 'converteu').sum():,} converteram; {(tr['desfecho'] == 'expirou').sum():,} expiraram; "
          f"{len(em_aberto)} em andamento")

testes em andamento: 470, iniciados a partir de 2026-09-24 06:10:39
assinaturas com canal diferente do teste: 0


desligou_renovacao_no_trial,False,True,total
desfecho,,,
converteu,27383,0,27383
em_andamento,448,22,470
expirou,12431,6712,19143
total,40262,6734,46996


[trials] 8. Uma linha por teste grátis: 46,996 linhas (27,383 converteram; 19,143 expiraram; 470 em andamento)


## 9. Saídas e integridade dos brutos

In [13]:
COLS_PAINEL = ["mes", "assinatura_id", "user_id", "plano_atual", "canal_cobranca", "inicio_antes_do_log",
               "meses_de_casa", "ativa_inicio_mes", "iniciou_no_mes", "renovacao_ligada_inicio_mes",
               "encerrou_no_mes", "perdida_no_mes", "motivo_fim"] + sorted(cols_n)
painel_out = painel[COLS_PAINEL].rename(columns={"plano_atual": "plano"})
painel_out["mes"] = painel_out["mes"].dt.to_timestamp()

saidas = {
    "assinaturas_mes.parquet": painel_out,
    "assinaturas_resumo_eventos.parquet": resumo,
    "trials.parquet": tr,
}
for nome, df in saidas.items():
    for c in df.select_dtypes("object").columns:   # ausente sempre como None nas colunas de texto
        df[c] = df[c].astype(object).where(df[c].notna(), None)
    df.to_parquet(PROCESSADOS / nome, index=False)
    pd.testing.assert_frame_equal(pd.read_parquet(PROCESSADOS / nome), df.reset_index(drop=True))
    print(f"dados/processados/{nome}: {len(df):,} linhas x {df.shape[1]} colunas")

pd.DataFrame(registro).to_csv(SAIDAS / "limpeza_eventos_assinatura_registro.csv", index=False)
display(pd.DataFrame(registro))

hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados/processados/assinaturas_mes.parquet: 327,771 linhas x 22 colunas
dados/processados/assinaturas_resumo_eventos.parquet: 32,583 linhas x 22 colunas


dados/processados/trials.parquet: 46,996 linhas x 11 colunas


,tabela,passo,linhas,observacao
0,log,0. Arquivo bruto,275565,
1,log,1. Data em Brasília,275565,"21,951 mudam de dia; 0 de contas de teste"
2,log,2. Eventos do teste grátis marcados,275565,"72,873 do trial; 202,692 da assinatura"
3,log,3. Assinatura de cada evento,275565,398 avisos atrasados contam no mês da perda
4,log,4. eventos_assinatura_limpo.parquet,275565,
5,painel,6.1 Uma linha por assinatura e mês,327771,"esperado 327,771"
6,painel,6.2 Situação no mês e eventos,327771,"202,692 eventos da assinatura distribuídos, to..."
7,resumo,7. Uma linha por assinatura,32583,
8,trials,8. Uma linha por teste grátis,46996,"27,383 converteram; 19,143 expiraram; 470 em a..."


dados/brutos/ inalterado: True
